# 03 – Gemischte Daten mit ColumnTransformer
Ein kontrollierter Kundendatensatz enthält Zahlen, Kategorien und fehlende Werte. Der ColumnTransformer verarbeitet jede Spaltenart passend; die Pipeline verhindert Leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

## Kontrollierte Rohdaten

In [ ]:
rng=np.random.default_rng(42); n=800
alter=rng.normal(42,13,n).clip(18,80); einkommen=rng.lognormal(10.7,.45,n); besuche=rng.poisson(5,n)
stadt=rng.choice(["Berlin","Hamburg","München"],n,p=[.45,.3,.25]); vertrag=rng.choice(["Monat","Jahr"],n,p=[.6,.4])
logit=-2+.035*(alter-40)+.000018*(einkommen-45000)+.16*besuche+.7*(vertrag=="Jahr")+.35*(stadt=="München")
p=1/(1+np.exp(-logit)); y=pd.Series(rng.binomial(1,p),name="kauft")
X=pd.DataFrame({"alter":alter,"einkommen":einkommen,"besuche":besuche,"stadt":stadt,"vertrag":vertrag})
X.loc[rng.choice(n,55,replace=False),"alter"]=np.nan; X.loc[rng.choice(n,40,replace=False),"einkommen"]=np.nan; X.loc[rng.choice(n,25,replace=False),"stadt"]=np.nan
display(X.head()); print(X.isna().sum()); print(y.value_counts())

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4)); X["alter"].hist(ax=axes[0],bins=25); axes[0].set_title("Alter"); pd.crosstab(X["vertrag"],y,normalize="index")[1].plot.bar(ax=axes[1]); axes[1].set_title("Kaufanteil nach Vertrag"); axes[1].set_ylabel("Anteil"); plt.tight_layout(); plt.show()

## Split vor der Vorverarbeitung

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
numerisch=["alter","einkommen","besuche"]; kategorisch=["stadt","vertrag"]
num_pipe=Pipeline([('imputer',SimpleImputer(strategy="median")),('scaler',StandardScaler())])
kat_pipe=Pipeline([('imputer',SimpleImputer(strategy="most_frequent")),('onehot',OneHotEncoder(handle_unknown="ignore"))])
preprocessing=ColumnTransformer([('num',num_pipe,numerisch),('kat',kat_pipe,kategorisch)])
modell=Pipeline([('preprocessing',preprocessing),('modell',LogisticRegression(max_iter=1000))])
modell.fit(X_train,y_train)

## Was hat die Vorverarbeitung erzeugt?

In [ ]:
namen=modell.named_steps["preprocessing"].get_feature_names_out()
transformiert=modell.named_steps["preprocessing"].transform(X_train.head())
print("Modellmatrix:",transformiert.shape); print(namen); display(pd.DataFrame(transformiert,columns=namen,index=X_train.head().index))

## Bewertung und neue Rohdaten

In [ ]:
y_pred=modell.predict(X_test); print(classification_report(y_test,y_pred)); ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap="Blues"); plt.show()
neu=pd.DataFrame({"alter":[np.nan,55],"einkommen":[52000,78000],"besuche":[4,9],"stadt":["Kiel","Berlin"],"vertrag":["Monat","Jahr"]})
display(neu.assign(kaufwahrscheinlichkeit=modell.predict_proba(neu)[:,1]))

Die Pipeline behandelt fehlende Werte und sogar die im Training unbekannte Stadt Kiel. Alle Regeln stammen ausschließlich aus den Trainingsdaten und werden bei neuen Daten identisch angewendet.